In [87]:
import os, sys, numpy
from typing import Any
from pandas import Series, DataFrame
from pandas import Timestamp, Timedelta
from pandas import read_pickle, concat
from pandas import Index, MultiIndex
from tqdm import tqdm

tests = dict[str, Any]()
for fn in os.listdir("../tests/"):
    if fn.endswith(".pickle"):
        key = fn[: 14]
        if key not in tests:
            tests[key] = list[str]()
        list.append(tests[key], fn)

print("Available tests:")
for key, files in tests.items():
    print(f" >> {key}: ({len(files)} files) ... {files}")
file_keys = ["ticks_p", "candles_p", "ticks_b", "candles_b"]

Available tests:
 >> 20260513035122: (4 files) ... ['20260513035122_candles_b.pickle', '20260513035122_candles_p.pickle', '20260513035122_ticks_b.pickle', '20260513035122_ticks_p.pickle']
 >> 20260513035123: (4 files) ... ['20260513035123_candles_b.pickle', '20260513035123_candles_p.pickle', '20260513035123_ticks_b.pickle', '20260513035123_ticks_p.pickle']
 >> 20260513035124: (4 files) ... ['20260513035124_candles_b.pickle', '20260513035124_candles_p.pickle', '20260513035124_ticks_b.pickle', '20260513035124_ticks_p.pickle']
 >> 20260513035126: (4 files) ... ['20260513035126_candles_b.pickle', '20260513035126_candles_p.pickle', '20260513035126_ticks_b.pickle', '20260513035126_ticks_p.pickle']
 >> 20260513035127: (4 files) ... ['20260513035127_candles_b.pickle', '20260513035127_candles_p.pickle', '20260513035127_ticks_b.pickle', '20260513035127_ticks_p.pickle']
 >> 20260513035134: (4 files) ... ['20260513035134_candles_b.pickle', '20260513035134_candles_p.pickle', '20260513035134_ticks_b

In [ ]:
side = "b"
columns = Index([*"ohlc"])
stats = dict()
ccount = dict()
iterator = tqdm(tests.items())
for key, files in iterator:
    ccount[key] = dict()
    iterator.set_description(f"Processing {key}...")
    dft = read_pickle(f"../tests/{key}_ticks_b.pickle")
    dfc = read_pickle(f"../tests/{key}_candles_b.pickle")
    dfp = read_pickle(f"../tests/{key}_candles_p.pickle")
    dft = dft.reset_index(["venue"], drop = True)
    dfc = dfc.reset_index(["venue"], drop = True)
    dfp = dfp.reset_index(["venue"], drop = True)
    dft = dft.reset_index("time")
    dft["time"] = dft["time"].dt.floor("1s")
    dft = dft.groupby(["symbol", "time"])["p" + side].apply(list)
    dfc = dfc[[*(columns + side), "volume"]]
    dfp = dfp[[*(columns + side), "volume"]]
    dfc.columns, dfp.columns = [*columns, "v"], [*columns, "v"]
    dfc = dfc.rename_axis("field", axis = "columns")
    dfp = dfp.rename_axis("field", axis = "columns")
    
    for tf in dfp.index.get_level_values("tf").unique():
        nc = dfc.query("tf == @tf").shape[0]
        np = dfp.query("tf == @tf").shape[0]
        ccount[key][tf] = {"b": nc, "p": np, "dt": Timedelta(
                tf[1 :] + tf[0].lower().replace("m", "min"))}
    dfc = dfc.xs("S1", level = "tf", drop_level = True)
    dfp = dfp.xs("S1", level = "tf", drop_level = True)

    comp = DataFrame.merge(dfc.stack("field").rename("b"), dfp.stack("field").rename("p"),
            left_index = True, right_index = True, how = "outer").sort_index().dropna()
    stat: Series = comp["b"] / comp["p"] - 1
    stat = stat.groupby(["symbol", "field"]).describe()
    stats[key] = stat

stats = concat(stats, names = ["test", "symbol", "field"])
ccount = DataFrame.from_dict(ccount, orient = "index")
ccount = ccount.stack().apply(Series)
ccount = ccount.rename_axis(["key", "tf"])
ccount["diff"] = ccount["p"] - ccount["b"]
ccount["pat"] = ccount.apply("{0[b]}/{0[p]}".format, axis = "columns")
# ccount["pat"].unstack("tf").sort_index().transpose()

Processing 20260513040042...: 100%|██████████| 16/16 [00:29<00:00,  1.83s/it]


In [100]:
ccount.xs("S5", level = "tf", drop_level = True)

,b,p,dt,diff,pat
key,,,,,
20260513035122,240,481,0 days 00:00:05,241,240/481
20260513035123,240,241,0 days 00:00:05,1,240/241
20260513035124,120,121,0 days 00:00:05,1,120/121
20260513035126,48,49,0 days 00:00:05,1,48/49
20260513035127,480,962,0 days 00:00:05,482,480/962
20260513035134,480,482,0 days 00:00:05,2,480/482
20260513035140,240,242,0 days 00:00:05,2,240/242
20260513035145,96,98,0 days 00:00:05,2,96/98
20260513035149,24000,48001,0 days 00:00:05,24001,24000/48001
